In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# TASK 3 — DEEP LEARNING RECOMMENDATION MODEL (DLRM)
# ============================================================

import gc
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.calibration import calibration_curve


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 2. PATHS
# ============================================================

TRAIN_PATH = (
    "/kaggle/input/datasets/chinmaysharma147/"
    "dataset-task2/train.csv"
)

TEST_PATH = (
    "/kaggle/input/datasets/chinmaysharma147/"
    "dataset-task2/test.csv"
)


# ============================================================
# 3. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print(train_df.head())


# ============================================================
# 4. FEATURE IDENTIFICATION
# ============================================================

TARGET = "label"

# ------------------------------------------------------------
# IMPORTANT:
#
# If your dataset has explicit feature names, preferably put
# the exact 13 numerical and 26 categorical columns here.
#
# Example:
#
# NUMERICAL_FEATURES = ["I1", "I2", ..., "I13"]
# CATEGORICAL_FEATURES = ["C1", "C2", ..., "C26"]
#
# ------------------------------------------------------------

ALL_FEATURES = [
    c for c in train_df.columns
    if c != TARGET
]

# Automatic fallback:
# numerical columns are those detected as numeric.
numeric_candidates = [
    c for c in ALL_FEATURES
    if pd.api.types.is_numeric_dtype(train_df[c])
]

categorical_candidates = [
    c for c in ALL_FEATURES
    if c not in numeric_candidates
]

# If your categorical columns are integer encoded in the CSV,
# the automatic dtype method may classify them incorrectly.
# In that case, REPLACE the following two assignments manually.

NUMERICAL_FEATURES = numeric_candidates
CATEGORICAL_FEATURES = categorical_candidates

print("\nNumerical features:", len(NUMERICAL_FEATURES))
print(NUMERICAL_FEATURES)

print("\nCategorical features:", len(CATEGORICAL_FEATURES))
print(CATEGORICAL_FEATURES)

assert len(NUMERICAL_FEATURES) == 13, (
    f"Expected 13 numerical features, "
    f"found {len(NUMERICAL_FEATURES)}"
)

assert len(CATEGORICAL_FEATURES) == 26, (
    f"Expected 26 categorical features, "
    f"found {len(CATEGORICAL_FEATURES)}"
)


# ============================================================
# 5. TARGET
# ============================================================

y = train_df[TARGET].astype(np.float32).values

X_train_df = train_df.drop(columns=[TARGET]).copy()

if TARGET in test_df.columns:
    y_test = test_df[TARGET].astype(np.float32).values
    X_test_df = test_df.drop(columns=[TARGET]).copy()
else:
    y_test = None
    X_test_df = test_df.copy()


# ============================================================
# 6. LEAKAGE-SAFE PREPROCESSING
# ============================================================

def fit_preprocessors(train_part):

    # ---------------- Numerical ----------------

    num_imputer = SimpleImputer(strategy="median")

    num_scaler = StandardScaler()

    num_values = num_imputer.fit_transform(
        train_part[NUMERICAL_FEATURES]
    )

    num_values = num_scaler.fit_transform(num_values)

    # ---------------- Categorical ----------------

    cat_imputer = SimpleImputer(
        strategy="constant",
        fill_value="MISSING"
    )

    cat_values = cat_imputer.fit_transform(
        train_part[CATEGORICAL_FEATURES]
    )

    cat_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    cat_values = cat_encoder.fit_transform(cat_values)

    # Shift by one:
    #
    # unknown = 0
    # known categories = 1,2,3,...
    #
    cat_values = cat_values.astype(np.int64) + 1

    vocab_sizes = [
        len(categories) + 1
        for categories in cat_encoder.categories_
    ]

    preprocessors = {
        "num_imputer": num_imputer,
        "num_scaler": num_scaler,
        "cat_imputer": cat_imputer,
        "cat_encoder": cat_encoder,
        "vocab_sizes": vocab_sizes
    }

    return preprocessors


def transform_data(df, preprocessors):

    num_values = preprocessors["num_imputer"].transform(
        df[NUMERICAL_FEATURES]
    )

    num_values = preprocessors["num_scaler"].transform(
        num_values
    )

    cat_values = preprocessors["cat_imputer"].transform(
        df[CATEGORICAL_FEATURES]
    )

    cat_values = preprocessors["cat_encoder"].transform(
        cat_values
    )

    cat_values = cat_values.astype(np.int64) + 1

    return (
        num_values.astype(np.float32),
        cat_values.astype(np.int64)
    )


# ============================================================
# 7. MLP
# ============================================================

class MLP(nn.Module):

    def __init__(
        self,
        layer_sizes,
        activation_class=nn.ReLU
    ):

        super().__init__()

        layers = []

        for i in range(len(layer_sizes) - 1):

            layers.append(
                nn.Linear(
                    layer_sizes[i],
                    layer_sizes[i + 1]
                )
            )

            # No activation after final layer
            if i < len(layer_sizes) - 2:
                layers.append(
                    activation_class()
                )

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# 8. DLRM MODEL
# ============================================================

class DLRM(nn.Module):

    def __init__(
        self,
        num_dense_features,
        vocab_sizes,
        embed_dim=16,
        bottom_mlp_sizes=(64, 32),
        top_mlp_sizes=(128, 64),
        activation_class=nn.ReLU
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.num_sparse_features = len(vocab_sizes)

        # ----------------------------------------------------
        # Embedding tables
        # ----------------------------------------------------

        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embed_dim
            )
            for vocab_size in vocab_sizes
        ])

        # ----------------------------------------------------
        # Bottom MLP
        #
        # 13 numerical features
        #       ↓
        #      MLP
        #       ↓
        # embed_dim
        # ----------------------------------------------------

        bottom_sizes = (
            [num_dense_features]
            + list(bottom_mlp_sizes)
            + [embed_dim]
        )

        self.bottom_mlp = MLP(
            bottom_sizes,
            activation_class
        )

        # ----------------------------------------------------
        # Number of feature vectors
        #
        # 1 dense representation
        # + 26 categorical embeddings
        # = 27 vectors
        # ----------------------------------------------------

        num_feature_vectors = (
            self.num_sparse_features + 1
        )

        # Number of pairwise interactions
        num_interactions = (
            num_feature_vectors
            * (num_feature_vectors - 1)
            // 2
        )

        # DLRM retains the dense vector and appends
        # pairwise interaction values.
        top_input_dim = (
            embed_dim + num_interactions
        )

        print(
            "DLRM interaction dimension:",
            num_interactions
        )

        print(
            "Top MLP input dimension:",
            top_input_dim
        )

        # ----------------------------------------------------
        # Top MLP
        # ----------------------------------------------------

        top_sizes = (
            [top_input_dim]
            + list(top_mlp_sizes)
            + [1]
        )

        self.top_mlp = MLP(
            top_sizes,
            activation_class
        )

    # --------------------------------------------------------
    # DLRM INTERACTION OPERATION
    # --------------------------------------------------------

    def interact(self, dense_vector, sparse_vectors):

        # feature_vectors:
        #
        # [dense_vector,
        #  embedding_1,
        #  embedding_2,
        #  ...]
        #
        feature_vectors = (
            [dense_vector]
            + sparse_vectors
        )

        interactions = []

        # Explicit pairwise dot products
        for i in range(len(feature_vectors)):

            for j in range(i + 1, len(feature_vectors)):

                interaction = (
                    feature_vectors[i]
                    * feature_vectors[j]
                ).sum(dim=1, keepdim=True)

                interactions.append(interaction)

        # Concatenate all pairwise interactions
        interaction_vector = torch.cat(
            interactions,
            dim=1
        )

        # DLRM concatenates dense representation
        # with interaction results.
        output = torch.cat(
            [dense_vector, interaction_vector],
            dim=1
        )

        return output

    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    def forward(self, dense_x, sparse_x):

        # -----------------------------
        # Dense pathway
        # -----------------------------

        dense_vector = self.bottom_mlp(
            dense_x
        )

        # -----------------------------
        # Sparse pathway
        # -----------------------------

        sparse_vectors = []

        for i, embedding in enumerate(
            self.embeddings
        ):

            sparse_vectors.append(
                embedding(
                    sparse_x[:, i]
                )
            )

        # -----------------------------
        # Interaction
        # -----------------------------

        interaction_vector = self.interact(
            dense_vector,
            sparse_vectors
        )

        # -----------------------------
        # Top MLP
        # -----------------------------

        logits = self.top_mlp(
            interaction_vector
        )

        # Return logits, NOT sigmoid
        #
        # BCEWithLogitsLoss will handle
        # the sigmoid internally.
        # -----------------------------

        return logits.squeeze(1)


# ============================================================
# 9. PARAMETER COUNT
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# ============================================================
# 10. TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    learning_rate=1e-3,
    weight_decay=0.0,
    epochs=30,
    patience=4
):

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None

    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": []
    }

    for epoch in range(epochs):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        train_loss = 0.0
        train_count = 0

        for dense_batch, sparse_batch, y_batch in train_loader:

            dense_batch = dense_batch.to(DEVICE)
            sparse_batch = sparse_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            logits = model(
                dense_batch,
                sparse_batch
            )

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                * len(y_batch)
            )

            train_count += len(y_batch)

        train_loss /= train_count

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_loss = 0.0
        val_count = 0

        val_probs = []
        val_targets = []

        with torch.no_grad():

            for dense_batch, sparse_batch, y_batch in val_loader:

                dense_batch = dense_batch.to(DEVICE)
                sparse_batch = sparse_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                logits = model(
                    dense_batch,
                    sparse_batch
                )

                loss = criterion(
                    logits,
                    y_batch
                )

                probabilities = torch.sigmoid(
                    logits
                )

                val_loss += (
                    loss.item()
                    * len(y_batch)
                )

                val_count += len(y_batch)

                val_probs.extend(
                    probabilities.cpu().numpy()
                )

                val_targets.extend(
                    y_batch.cpu().numpy()
                )

        val_loss /= val_count

        val_auc = roc_auc_score(
            val_targets,
            val_probs
        )

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_auc"].append(
            val_auc
        )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Val Loss: {val_loss:.5f} | "
            f"Val AUC: {val_auc:.5f}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

            if patience_counter >= patience:

                print("Early stopping.")

                break

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return model, history


# ============================================================
# 11. PREDICTION
# ============================================================

def predict_model(
    model,
    loader
):

    model.eval()

    probabilities = []

    targets = []

    with torch.no_grad():

        for dense_batch, sparse_batch, y_batch in loader:

            dense_batch = dense_batch.to(DEVICE)
            sparse_batch = sparse_batch.to(DEVICE)

            logits = model(
                dense_batch,
                sparse_batch
            )

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            targets.extend(
                y_batch.numpy()
            )

    return (
        np.asarray(targets),
        np.asarray(probabilities)
    )


# ============================================================
# 12. METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_prob,
    threshold=0.5
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob
        ),

        "PR-AUC": average_precision_score(
            y_true,
            y_prob
        ),

        "Log Loss": log_loss(
            y_true,
            np.clip(y_prob, 1e-7, 1 - 1e-7)
        ),

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    }


# ============================================================
# 13. THRESHOLD SEARCH
# ============================================================

def find_best_threshold(
    y_true,
    y_prob
):

    thresholds = np.arange(
        0.05,
        0.96,
        0.01
    )

    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:

        predictions = (
            y_prob >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = threshold

    return best_threshold


# ============================================================
# 14. 5-FOLD DLRM CROSS VALIDATION
# ============================================================

param_grid = {

    "embed_dim": [16, 32],

    "bottom_mlp_sizes": [
        (64, 32),
        (128, 64)
    ],

    "top_mlp_sizes": [
        (128, 64)
    ],

    "learning_rate": [
        1e-3,
        1e-4
    ],

    "activation_class": [
        nn.ReLU,
        nn.GELU
    ],

    "batch_size": [
        256
    ]
}


grid = list(
    ParameterGrid(param_grid)
)

print(
    "\nTotal hyperparameter combinations:",
    len(grid)
)


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv_results = []


for combo_number, params in enumerate(
    grid,
    start=1
):

    print("\n" + "=" * 70)

    print(
        f"Combination {combo_number}/{len(grid)}"
    )

    print(params)

    fold_auc = []
    fold_logloss = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(
            X_train_df,
            y
        ),
        start=1
    ):

        print(
            f"\nFold {fold}/5"
        )

        # ----------------------------------------------------
        # Split raw data FIRST
        # ----------------------------------------------------

        fold_train = X_train_df.iloc[
            train_idx
        ]

        fold_val = X_train_df.iloc[
            val_idx
        ]

        y_fold_train = y[
            train_idx
        ]

        y_fold_val = y[
            val_idx
        ]

        # ----------------------------------------------------
        # Fit preprocessing ONLY on fold training data
        # ----------------------------------------------------

        preprocessors = fit_preprocessors(
            fold_train
        )

        X_num_train, X_cat_train = (
            transform_data(
                fold_train,
                preprocessors
            )
        )

        X_num_val, X_cat_val = (
            transform_data(
                fold_val,
                preprocessors
            )
        )

        # ----------------------------------------------------
        # Tensor datasets
        # ----------------------------------------------------

        train_dataset = TensorDataset(
            torch.tensor(
                X_num_train,
                dtype=torch.float32
            ),

            torch.tensor(
                X_cat_train,
                dtype=torch.long
            ),

            torch.tensor(
                y_fold_train,
                dtype=torch.float32
            )
        )

        val_dataset = TensorDataset(
            torch.tensor(
                X_num_val,
                dtype=torch.float32
            ),

            torch.tensor(
                X_cat_val,
                dtype=torch.long
            ),

            torch.tensor(
                y_fold_val,
                dtype=torch.float32
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=params["batch_size"],
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=params["batch_size"],
            shuffle=False
        )

        # ----------------------------------------------------
        # Build model
        # ----------------------------------------------------

        model = DLRM(

            num_dense_features=len(
                NUMERICAL_FEATURES
            ),

            vocab_sizes=preprocessors[
                "vocab_sizes"
            ],

            embed_dim=params[
                "embed_dim"
            ],

            bottom_mlp_sizes=params[
                "bottom_mlp_sizes"
            ],

            top_mlp_sizes=params[
                "top_mlp_sizes"
            ],

            activation_class=params[
                "activation_class"
            ]

        ).to(DEVICE)

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model, history = train_model(

            model,

            train_loader,

            val_loader,

            learning_rate=params[
                "learning_rate"
            ],

            epochs=20,

            patience=3
        )

        # ----------------------------------------------------
        # Validation prediction
        # ----------------------------------------------------

        y_val_true, y_val_prob = predict_model(
            model,
            val_loader
        )

        auc = roc_auc_score(
            y_val_true,
            y_val_prob
        )

        ll = log_loss(
            y_val_true,
            np.clip(
                y_val_prob,
                1e-7,
                1 - 1e-7
            )
        )

        fold_auc.append(auc)
        fold_logloss.append(ll)

        print(
            f"Fold ROC-AUC: {auc:.5f}"
        )

        print(
            f"Fold Log Loss: {ll:.5f}"
        )

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Mean CV result
    # --------------------------------------------------------

    mean_auc = np.mean(
        fold_auc
    )

    mean_logloss = np.mean(
        fold_logloss
    )

    cv_results.append({

        **params,

        "mean_roc_auc": mean_auc,

        "mean_log_loss": mean_logloss
    })

    print(
        "\nMean ROC-AUC:",
        mean_auc
    )

    print(
        "Mean Log Loss:",
        mean_logloss
    )


# ============================================================
# 15. FIND BEST PARAMETERS
# ============================================================

results_df = pd.DataFrame(
    cv_results
)

results_df = results_df.sort_values(
    "mean_roc_auc",
    ascending=False
)

print(
    "\n===== CV RESULTS ====="
)

print(
    results_df.head(10)
)

best_params = results_df.iloc[0].to_dict()

print(
    "\n===== BEST PARAMETERS ====="
)

print(best_params)


# ============================================================
# 16. FINAL PREPROCESSING
# ============================================================

final_preprocessors = fit_preprocessors(
    X_train_df
)

X_num_train, X_cat_train = transform_data(
    X_train_df,
    final_preprocessors
)

X_num_test, X_cat_test = transform_data(
    X_test_df,
    final_preprocessors
)


# ============================================================
# 17. FINAL MODEL
# ============================================================

final_model = DLRM(

    num_dense_features=len(
        NUMERICAL_FEATURES
    ),

    vocab_sizes=final_preprocessors[
        "vocab_sizes"
    ],

    embed_dim=int(
        best_params["embed_dim"]
    ),

    bottom_mlp_sizes=best_params[
        "bottom_mlp_sizes"
    ],

    top_mlp_sizes=best_params[
        "top_mlp_sizes"
    ],

    activation_class=(
        nn.ReLU
        if best_params["activation_class"] == nn.ReLU
        else nn.GELU
    )

).to(DEVICE)


# ============================================================
# 18. PARAMETER COUNT
# ============================================================

total_params = count_parameters(
    final_model
)

print(
    "\nTotal trainable parameters:",
    f"{total_params:,}"
)

print(
    "Approximate FP32 parameter memory:",
    f"{total_params * 4 / (1024 ** 2):.2f} MB"
)


# ============================================================
# 19. FINAL TRAINING
# ============================================================

train_dataset = TensorDataset(

    torch.tensor(
        X_num_train,
        dtype=torch.float32
    ),

    torch.tensor(
        X_cat_train,
        dtype=torch.long
    ),

    torch.tensor(
        y,
        dtype=torch.float32
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)


# Since the final model is being trained on all
# available training data, we don't use test data
# for early stopping.

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    final_model.parameters(),
    lr=float(
        best_params["learning_rate"]
    )
)


EPOCHS = 20

start_time = time.time()

final_model.train()

for epoch in range(EPOCHS):

    epoch_loss = 0.0

    for dense_batch, sparse_batch, y_batch in train_loader:

        dense_batch = dense_batch.to(DEVICE)
        sparse_batch = sparse_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = final_model(
            dense_batch,
            sparse_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        epoch_loss += (
            loss.item()
            * len(y_batch)
        )

    epoch_loss /= len(
        train_dataset
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {epoch_loss:.5f}"
    )


training_time = time.time() - start_time

print(
    "\nFinal training time:",
    f"{training_time:.2f} seconds"
)


# ============================================================
# 20. TEST PREDICTION
# ============================================================


test_dataset = TensorDataset(
    torch.tensor(X_num_test, dtype=torch.float32),
    torch.tensor(X_cat_test, dtype=torch.long)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)


final_model.eval()

test_probabilities = []

with torch.no_grad():

    for dense_batch, sparse_batch, _ in test_loader:

        dense_batch = dense_batch.to(DEVICE)
        sparse_batch = sparse_batch.to(DEVICE)

        logits = final_model(
            dense_batch,
            sparse_batch
        )

        probabilities = torch.sigmoid(
            logits
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )


test_probabilities = np.asarray(
    test_probabilities
)


# ============================================================
# 21. TEST METRICS
# ============================================================

if y_test is not None:

    # Default threshold
    test_metrics = calculate_metrics(
        y_test,
        test_probabilities,
        threshold=0.5
    )

    print(
        "\n===== TEST RESULTS @ 0.5 ====="
    )

    for metric, value in test_metrics.items():

        print(
            f"{metric}: {value:.5f}"
        )

else:

    print(
        "\nTest labels are not present."
    )


# ============================================================
# 22. CALIBRATION
# ============================================================

if y_test is not None:

    fraction_positive, mean_predicted = (
        calibration_curve(
            y_test,
            test_probabilities,
            n_bins=10,
            strategy="uniform"
        )
    )

    import matplotlib.pyplot as plt

    plt.figure(figsize=(7, 6))

    plt.plot(
        mean_predicted,
        fraction_positive,
        marker="o",
        label="DLRM"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.xlabel(
        "Mean predicted probability"
    )

    plt.ylabel(
        "Fraction of positives"
    )

    plt.title(
        "DLRM Reliability Diagram"
    )

    plt.legend()

    plt.grid(True)

    plt.show()

In [ ]:
#ABLATION-REMOVE THE INTERACTIONS OF SECOND ORDER(NEURAL NETWORK TAKES A LONG TIME TO EVALUATE THE INTERACTIONS AND REDUCES EFFICIENCY)
# ============================================================
# TASK 3 — DEEP LEARNING RECOMMENDATION MODEL (DLRM)
# ============================================================

import gc
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.calibration import calibration_curve


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 2. PATHS
# ============================================================

TRAIN_PATH = (
    "/kaggle/input/datasets/chinmaysharma147/"
    "dataset-task2/train.csv"
)

TEST_PATH = (
    "/kaggle/input/datasets/chinmaysharma147/"
    "dataset-task2/test.csv"
)


# ============================================================
# 3. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print(train_df.head())


# ============================================================
# 4. FEATURE IDENTIFICATION
# ============================================================

TARGET = "label"

# ------------------------------------------------------------
# IMPORTANT:
#
# If your dataset has explicit feature names, preferably put
# the exact 13 numerical and 26 categorical columns here.
#
# Example:
#
# NUMERICAL_FEATURES = ["I1", "I2", ..., "I13"]
# CATEGORICAL_FEATURES = ["C1", "C2", ..., "C26"]
#
# ------------------------------------------------------------

ALL_FEATURES = [
    c for c in train_df.columns
    if c != TARGET
]

# Automatic fallback:
# numerical columns are those detected as numeric.
numeric_candidates = [
    c for c in ALL_FEATURES
    if pd.api.types.is_numeric_dtype(train_df[c])
]

categorical_candidates = [
    c for c in ALL_FEATURES
    if c not in numeric_candidates
]

# If your categorical columns are integer encoded in the CSV,
# the automatic dtype method may classify them incorrectly.
# In that case, REPLACE the following two assignments manually.

NUMERICAL_FEATURES = numeric_candidates
CATEGORICAL_FEATURES = categorical_candidates

print("\nNumerical features:", len(NUMERICAL_FEATURES))
print(NUMERICAL_FEATURES)

print("\nCategorical features:", len(CATEGORICAL_FEATURES))
print(CATEGORICAL_FEATURES)

assert len(NUMERICAL_FEATURES) == 13, (
    f"Expected 13 numerical features, "
    f"found {len(NUMERICAL_FEATURES)}"
)

assert len(CATEGORICAL_FEATURES) == 26, (
    f"Expected 26 categorical features, "
    f"found {len(CATEGORICAL_FEATURES)}"
)


# ============================================================
# 5. TARGET
# ============================================================

y = train_df[TARGET].astype(np.float32).values

X_train_df = train_df.drop(columns=[TARGET]).copy()

if TARGET in test_df.columns:
    y_test = test_df[TARGET].astype(np.float32).values
    X_test_df = test_df.drop(columns=[TARGET]).copy()
else:
    y_test = None
    X_test_df = test_df.copy()


# ============================================================
# 6. LEAKAGE-SAFE PREPROCESSING
# ============================================================

def fit_preprocessors(train_part):

    # ---------------- Numerical ----------------

    num_imputer = SimpleImputer(strategy="median")

    num_scaler = StandardScaler()

    num_values = num_imputer.fit_transform(
        train_part[NUMERICAL_FEATURES]
    )

    num_values = num_scaler.fit_transform(num_values)

    # ---------------- Categorical ----------------

    cat_imputer = SimpleImputer(
        strategy="constant",
        fill_value="MISSING"
    )

    cat_values = cat_imputer.fit_transform(
        train_part[CATEGORICAL_FEATURES]
    )

    cat_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    cat_values = cat_encoder.fit_transform(cat_values)

    # Shift by one:
    #
    # unknown = 0
    # known categories = 1,2,3,...
    #
    cat_values = cat_values.astype(np.int64) + 1

    vocab_sizes = [
        len(categories) + 1
        for categories in cat_encoder.categories_
    ]

    preprocessors = {
        "num_imputer": num_imputer,
        "num_scaler": num_scaler,
        "cat_imputer": cat_imputer,
        "cat_encoder": cat_encoder,
        "vocab_sizes": vocab_sizes
    }

    return preprocessors


def transform_data(df, preprocessors):

    num_values = preprocessors["num_imputer"].transform(
        df[NUMERICAL_FEATURES]
    )

    num_values = preprocessors["num_scaler"].transform(
        num_values
    )

    cat_values = preprocessors["cat_imputer"].transform(
        df[CATEGORICAL_FEATURES]
    )

    cat_values = preprocessors["cat_encoder"].transform(
        cat_values
    )

    cat_values = cat_values.astype(np.int64) + 1

    return (
        num_values.astype(np.float32),
        cat_values.astype(np.int64)
    )


# ============================================================
# 7. MLP
# ============================================================

class MLP(nn.Module):

    def __init__(
        self,
        layer_sizes,
        activation_class=nn.ReLU
    ):

        super().__init__()

        layers = []

        for i in range(len(layer_sizes) - 1):

            layers.append(
                nn.Linear(
                    layer_sizes[i],
                    layer_sizes[i + 1]
                )
            )

            # No activation after final layer
            if i < len(layer_sizes) - 2:
                layers.append(
                    activation_class()
                )

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# 8. DLRM MODEL
# ============================================================

class DLRM(nn.Module):

    def __init__(
        self,
        num_dense_features,
        vocab_sizes,
        embed_dim=16,
        bottom_mlp_sizes=(64, 32),
        top_mlp_sizes=(128, 64),
        activation_class=nn.ReLU
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.num_sparse_features = len(vocab_sizes)

        # ----------------------------------------------------
        # Embedding tables
        # ----------------------------------------------------

        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embed_dim
            )
            for vocab_size in vocab_sizes
        ])

        # ----------------------------------------------------
        # Bottom MLP
        #
        # 13 numerical features
        #       ↓
        #      MLP
        #       ↓
        # embed_dim
        # ----------------------------------------------------

        bottom_sizes = (
            [num_dense_features]
            + list(bottom_mlp_sizes)
            + [embed_dim]
        )

        self.bottom_mlp = MLP(
            bottom_sizes,
            activation_class
        )

        # ----------------------------------------------------
        # Number of feature vectors
        #
        # 1 dense representation
        # + 26 categorical embeddings
        # = 27 vectors
        # ----------------------------------------------------

        num_feature_vectors = (
            self.num_sparse_features + 1
        )

        # Number of pairwise interactions
        num_interactions = (
            num_feature_vectors
            * (num_feature_vectors - 1)
            // 2
        )

        # DLRM retains the dense vector and appends
        # pairwise interaction values.
        top_input_dim = (
            embed_dim + num_interactions
        )

        print(
            "DLRM interaction dimension:",
            num_interactions
        )

        print(
            "Top MLP input dimension:",
            top_input_dim
        )

        # ----------------------------------------------------
        # Top MLP
        # ----------------------------------------------------

        top_sizes = (
            [top_input_dim]
            + list(top_mlp_sizes)
            + [1]
        )

        self.top_mlp = MLP(
            top_sizes,
            activation_class
        )

  """  # --------------------------------------------------------
    # DLRM INTERACTION OPERATION
    # --------------------------------------------------------

    def interact(self, dense_vector, sparse_vectors):

        # feature_vectors:
        #
        # [dense_vector,
        #  embedding_1,
        #  embedding_2,
        #  ...]
        #
        feature_vectors = (
            [dense_vector]
            + sparse_vectors
        )

        interactions = []

        # Explicit pairwise dot products
        for i in range(len(feature_vectors)):

            for j in range(i + 1, len(feature_vectors)):

                interaction = (
                    feature_vectors[i]
                    * feature_vectors[j]
                ).sum(dim=1, keepdim=True)

                interactions.append(interaction)

        # Concatenate all pairwise interactions
        interaction_vector = torch.cat(
            interactions,
            dim=1
        )

        # DLRM concatenates dense representation
        # with interaction results.
        output = torch.cat(
            [dense_vector, interaction_vector],
            dim=1
        )

        return output """

    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    def forward(self, dense_x, sparse_x):

        # -----------------------------
        # Dense pathway
        # -----------------------------

        dense_vector = self.bottom_mlp(
            dense_x
        )

        # -----------------------------
        # Sparse pathway
        # -----------------------------

        sparse_vectors = []

        for i, embedding in enumerate(
            self.embeddings
        ):

            sparse_vectors.append(
                embedding(
                    sparse_x[:, i]
                )
            )

        combined_vec=[dense_x + sparse_x]
        

        # -----------------------------
        # Top MLP
        # -----------------------------

        logits = self.top_mlp(
            combined
        )

        # Return logits, NOT sigmoid
        #
        # BCEWithLogitsLoss will handle
        # the sigmoid internally.
        # -----------------------------

        return logits.squeeze(1)


# ============================================================
# 9. PARAMETER COUNT
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# ============================================================
# 10. TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    learning_rate=1e-3,
    weight_decay=0.0,
    epochs=30,
    patience=4
):

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None

    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": []
    }

    for epoch in range(epochs):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        train_loss = 0.0
        train_count = 0

        for dense_batch, sparse_batch, y_batch in train_loader:

            dense_batch = dense_batch.to(DEVICE)
            sparse_batch = sparse_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            logits = model(
                dense_batch,
                sparse_batch
            )

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                * len(y_batch)
            )

            train_count += len(y_batch)

        train_loss /= train_count

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_loss = 0.0
        val_count = 0

        val_probs = []
        val_targets = []

        with torch.no_grad():

            for dense_batch, sparse_batch, y_batch in val_loader:

                dense_batch = dense_batch.to(DEVICE)
                sparse_batch = sparse_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                logits = model(
                    dense_batch,
                    sparse_batch
                )

                loss = criterion(
                    logits,
                    y_batch
                )

                probabilities = torch.sigmoid(
                    logits
                )

                val_loss += (
                    loss.item()
                    * len(y_batch)
                )

                val_count += len(y_batch)

                val_probs.extend(
                    probabilities.cpu().numpy()
                )

                val_targets.extend(
                    y_batch.cpu().numpy()
                )

        val_loss /= val_count

        val_auc = roc_auc_score(
            val_targets,
            val_probs
        )

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_auc"].append(
            val_auc
        )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Val Loss: {val_loss:.5f} | "
            f"Val AUC: {val_auc:.5f}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

            if patience_counter >= patience:

                print("Early stopping.")

                break

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return model, history


# ============================================================
# 11. PREDICTION
# ============================================================

def predict_model(
    model,
    loader
):

    model.eval()

    probabilities = []

    targets = []

    with torch.no_grad():

        for dense_batch, sparse_batch, y_batch in loader:

            dense_batch = dense_batch.to(DEVICE)
            sparse_batch = sparse_batch.to(DEVICE)

            logits = model(
                dense_batch,
                sparse_batch
            )

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            targets.extend(
                y_batch.numpy()
            )

    return (
        np.asarray(targets),
        np.asarray(probabilities)
    )


# ============================================================
# 12. METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_prob,
    threshold=0.5
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob
        ),

        "PR-AUC": average_precision_score(
            y_true,
            y_prob
        ),

        "Log Loss": log_loss(
            y_true,
            np.clip(y_prob, 1e-7, 1 - 1e-7)
        ),

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    }


# ============================================================
# 13. THRESHOLD SEARCH
# ============================================================

def find_best_threshold(
    y_true,
    y_prob
):

    thresholds = np.arange(
        0.05,
        0.96,
        0.01
    )

    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:

        predictions = (
            y_prob >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = threshold

    return best_threshold


# ============================================================
# 14. 5-FOLD DLRM CROSS VALIDATION
# ============================================================

param_grid = {

    "embed_dim": [16, 32],

    "bottom_mlp_sizes": [
        (64, 32),
        (128, 64)
    ],

    "top_mlp_sizes": [
        (128, 64)
    ],

    "learning_rate": [
        1e-3,
        1e-4
    ],

    "activation_class": [
        nn.ReLU,
        nn.GELU
    ],

    "batch_size": [
        256
    ]
}


grid = list(
    ParameterGrid(param_grid)
)

print(
    "\nTotal hyperparameter combinations:",
    len(grid)
)


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

cv_results = []


for combo_number, params in enumerate(
    grid,
    start=1
):

    print("\n" + "=" * 70)

    print(
        f"Combination {combo_number}/{len(grid)}"
    )

    print(params)

    fold_auc = []
    fold_logloss = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(
            X_train_df,
            y
        ),
        start=1
    ):

        print(
            f"\nFold {fold}/5"
        )

        # ----------------------------------------------------
        # Split raw data FIRST
        # ----------------------------------------------------

        fold_train = X_train_df.iloc[
            train_idx
        ]

        fold_val = X_train_df.iloc[
            val_idx
        ]

        y_fold_train = y[
            train_idx
        ]

        y_fold_val = y[
            val_idx
        ]

        # ----------------------------------------------------
        # Fit preprocessing ONLY on fold training data
        # ----------------------------------------------------

        preprocessors = fit_preprocessors(
            fold_train
        )

        X_num_train, X_cat_train = (
            transform_data(
                fold_train,
                preprocessors
            )
        )

        X_num_val, X_cat_val = (
            transform_data(
                fold_val,
                preprocessors
            )
        )

        # ----------------------------------------------------
        # Tensor datasets
        # ----------------------------------------------------

        train_dataset = TensorDataset(
            torch.tensor(
                X_num_train,
                dtype=torch.float32
            ),

            torch.tensor(
                X_cat_train,
                dtype=torch.long
            ),

            torch.tensor(
                y_fold_train,
                dtype=torch.float32
            )
        )

        val_dataset = TensorDataset(
            torch.tensor(
                X_num_val,
                dtype=torch.float32
            ),

            torch.tensor(
                X_cat_val,
                dtype=torch.long
            ),

            torch.tensor(
                y_fold_val,
                dtype=torch.float32
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=params["batch_size"],
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=params["batch_size"],
            shuffle=False
        )

        # ----------------------------------------------------
        # Build model
        # ----------------------------------------------------

        model = DLRM(

            num_dense_features=len(
                NUMERICAL_FEATURES
            ),

            vocab_sizes=preprocessors[
                "vocab_sizes"
            ],

            embed_dim=params[
                "embed_dim"
            ],

            bottom_mlp_sizes=params[
                "bottom_mlp_sizes"
            ],

            top_mlp_sizes=params[
                "top_mlp_sizes"
            ],

            activation_class=params[
                "activation_class"
            ]

        ).to(DEVICE)

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model, history = train_model(

            model,

            train_loader,

            val_loader,

            learning_rate=params[
                "learning_rate"
            ],

            epochs=20,

            patience=3
        )

        # ----------------------------------------------------
        # Validation prediction
        # ----------------------------------------------------

        y_val_true, y_val_prob = predict_model(
            model,
            val_loader
        )

        auc = roc_auc_score(
            y_val_true,
            y_val_prob
        )

        ll = log_loss(
            y_val_true,
            np.clip(
                y_val_prob,
                1e-7,
                1 - 1e-7
            )
        )

        fold_auc.append(auc)
        fold_logloss.append(ll)

        print(
            f"Fold ROC-AUC: {auc:.5f}"
        )

        print(
            f"Fold Log Loss: {ll:.5f}"
        )

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Mean CV result
    # --------------------------------------------------------

    mean_auc = np.mean(
        fold_auc
    )

    mean_logloss = np.mean(
        fold_logloss
    )

    cv_results.append({

        **params,

        "mean_roc_auc": mean_auc,

        "mean_log_loss": mean_logloss
    })

    print(
        "\nMean ROC-AUC:",
        mean_auc
    )

    print(
        "Mean Log Loss:",
        mean_logloss
    )


# ============================================================
# 15. FIND BEST PARAMETERS
# ============================================================

results_df = pd.DataFrame(
    cv_results
)

results_df = results_df.sort_values(
    "mean_roc_auc",
    ascending=False
)

print(
    "\n===== CV RESULTS ====="
)

print(
    results_df.head(10)
)

best_params = results_df.iloc[0].to_dict()

print(
    "\n===== BEST PARAMETERS ====="
)

print(best_params)


# ============================================================
# 16. FINAL PREPROCESSING
# ============================================================

final_preprocessors = fit_preprocessors(
    X_train_df
)

X_num_train, X_cat_train = transform_data(
    X_train_df,
    final_preprocessors
)

X_num_test, X_cat_test = transform_data(
    X_test_df,
    final_preprocessors
)


# ============================================================
# 17. FINAL MODEL
# ============================================================

final_model = DLRM(

    num_dense_features=len(
        NUMERICAL_FEATURES
    ),

    vocab_sizes=final_preprocessors[
        "vocab_sizes"
    ],

    embed_dim=int(
        best_params["embed_dim"]
    ),

    bottom_mlp_sizes=best_params[
        "bottom_mlp_sizes"
    ],

    top_mlp_sizes=best_params[
        "top_mlp_sizes"
    ],

    activation_class=(
        nn.ReLU
        if best_params["activation_class"] == nn.ReLU
        else nn.GELU
    )

).to(DEVICE)


# ============================================================
# 18. PARAMETER COUNT
# ============================================================

total_params = count_parameters(
    final_model
)

print(
    "\nTotal trainable parameters:",
    f"{total_params:,}"
)

print(
    "Approximate FP32 parameter memory:",
    f"{total_params * 4 / (1024 ** 2):.2f} MB"
)


# ============================================================
# 19. FINAL TRAINING
# ============================================================

train_dataset = TensorDataset(

    torch.tensor(
        X_num_train,
        dtype=torch.float32
    ),

    torch.tensor(
        X_cat_train,
        dtype=torch.long
    ),

    torch.tensor(
        y,
        dtype=torch.float32
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)


# Since the final model is being trained on all
# available training data, we don't use test data
# for early stopping.

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    final_model.parameters(),
    lr=float(
        best_params["learning_rate"]
    )
)


EPOCHS = 20

start_time = time.time()

final_model.train()

for epoch in range(EPOCHS):

    epoch_loss = 0.0

    for dense_batch, sparse_batch, y_batch in train_loader:

        dense_batch = dense_batch.to(DEVICE)
        sparse_batch = sparse_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = final_model(
            dense_batch,
            sparse_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        epoch_loss += (
            loss.item()
            * len(y_batch)
        )

    epoch_loss /= len(
        train_dataset
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {epoch_loss:.5f}"
    )


training_time = time.time() - start_time

print(
    "\nFinal training time:",
    f"{training_time:.2f} seconds"
)


# ============================================================
# 20. TEST PREDICTION
# ============================================================


test_dataset = TensorDataset(
    torch.tensor(X_num_test, dtype=torch.float32),
    torch.tensor(X_cat_test, dtype=torch.long)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)


final_model.eval()

test_probabilities = []

with torch.no_grad():

    for dense_batch, sparse_batch, _ in test_loader:

        dense_batch = dense_batch.to(DEVICE)
        sparse_batch = sparse_batch.to(DEVICE)

        logits = final_model(
            dense_batch,
            sparse_batch
        )

        probabilities = torch.sigmoid(
            logits
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )


test_probabilities = np.asarray(
    test_probabilities
)


# ============================================================
# 21. TEST METRICS
# ============================================================

if y_test is not None:

    # Default threshold
    test_metrics = calculate_metrics(
        y_test,
        test_probabilities,
        threshold=0.5
    )

    print(
        "\n===== TEST RESULTS @ 0.5 ====="
    )

    for metric, value in test_metrics.items():

        print(
            f"{metric}: {value:.5f}"
        )

else:

    print(
        "\nTest labels are not present."
    )


# ============================================================
# 22. CALIBRATION
# ============================================================

if y_test is not None:

    fraction_positive, mean_predicted = (
        calibration_curve(
            y_test,
            test_probabilities,
            n_bins=10,
            strategy="uniform"
        )
    )

    import matplotlib.pyplot as plt

    plt.figure(figsize=(7, 6))

    plt.plot(
        mean_predicted,
        fraction_positive,
        marker="o",
        label="DLRM"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.xlabel(
        "Mean predicted probability"
    )

    plt.ylabel(
        "Fraction of positives"
    )

    plt.title(
        "DLRM Reliability Diagram"
    )

    plt.legend()

    plt.grid(True)

    plt.show()